In [1]:
import os

# Define the folder paths
folders = [
    "src/Airbnb/components",
    "src/Airbnb/pipelines",
    "notebooks",
    "artifacts",
    "logs",
    "templates"
]

# Create folders and __init__.py files
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    
    # Add __init__.py to all src folders
    if "src" in folder:
        init_path = os.path.join(folder, "__init__.py")
        with open(init_path, "w") as f:
            pass

# Create the root src __init__
with open("src/__init__.py", "w") as f:
    pass
    
print("Directory structure created successfully!")

Directory structure created successfully!


In [2]:
%%writefile setup.py
from setuptools import setup, find_packages

setup(
    name="Airbnb-Price-Prediction",
    version="0.0.1",
    author="Saish",
    packages=find_packages(),
    install_requires=[] 
)

Overwriting setup.py


In [3]:
%%writefile src/Airbnb/logger.py
import logging
import os
from datetime import datetime

LOG_FILE = f"{datetime.now().strftime('%m_%d_%Y_%H_%M_%S')}.log"
logs_path = os.path.join(os.getcwd(), "logs")
os.makedirs(logs_path, exist_ok=True)

LOG_FILE_PATH = os.path.join(logs_path, LOG_FILE)

logging.basicConfig(
    filename=LOG_FILE_PATH,
    format="[%(asctime)s] %(lineno)d %(name)s - %(levelname)s - %(message)s",
    level=logging.INFO,
)

Overwriting src/Airbnb/logger.py


In [4]:
%%writefile src/Airbnb/exception.py
import sys

def error_message_detail(error, error_detail: sys):
    _, _, exc_tb = error_detail.exc_info()
    file_name = exc_tb.tb_frame.f_code.co_filename
    return f"Error in script [{file_name}] line [{exc_tb.tb_lineno}]: {str(error)}"

class CustomException(Exception):
    def __init__(self, error_message, error_detail: sys):
        super().__init__(error_message)
        self.error_message = error_message_detail(error_message, error_detail)

    def __str__(self):
        return self.error_message

Overwriting src/Airbnb/exception.py


In [5]:
%%writefile requirements.txt
pandas
numpy
scikit-learn
matplotlib
seaborn
flask

Overwriting requirements.txt


In [6]:
from src.Airbnb.logger import logging
from src.Airbnb.exception import CustomException
import sys

try:
    logging.info("Testing modular logging from Jupyter!")
    # Deliberate ZeroDivisionError to test exception handling
    result = 10 / 0
except Exception as e:
    logging.info("Caught zero division error successfully.")
    custom_err = CustomException(e, sys)
    print("Sanity Check Passed!")
    print(custom_err)

Sanity Check Passed!
Error in script [C:\Users\saish\AppData\Local\Temp\ipykernel_27700\3353687248.py] line [8]: division by zero


In [7]:
%%writefile src/Airbnb/components/data_ingestion.py
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from dataclasses import dataclass

from src.Airbnb.logger import logging
from src.Airbnb.exception import CustomException

@dataclass
class DataIngestionConfig:
    train_data_path: str = os.path.join('artifacts', 'train.csv')
    test_data_path: str = os.path.join('artifacts', 'test.csv')
    raw_data_path: str = os.path.join('artifacts', 'data.csv')

class DataIngestion:
    def __init__(self):
        self.ingestion_config = DataIngestionConfig()

    def initiate_data_ingestion(self):
        logging.info("Entered the data ingestion method or component")
        try:
            df = pd.read_csv('notebooks/data/raw_data.csv')
            logging.info('Read the dataset as dataframe')

            os.makedirs(os.path.dirname(self.ingestion_config.train_data_path), exist_ok=True)

            df.to_csv(self.ingestion_config.raw_data_path, index=False, header=True)

            logging.info("Train test split initiated")
            train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

            train_set.to_csv(self.ingestion_config.train_data_path, index=False, header=True)
            test_set.to_csv(self.ingestion_config.test_data_path, index=False, header=True)

            logging.info("Ingestion of the data is completed")

            return (
                self.ingestion_config.train_data_path,
                self.ingestion_config.test_data_path
            )
        except Exception as e:
            raise CustomException(e, sys)

Overwriting src/Airbnb/components/data_ingestion.py


In [8]:
from src.Airbnb.components.data_ingestion import DataIngestion
from src.Airbnb.logger import logging

try:
    logging.info("Starting Data Ingestion test from Notebook...")
    
    ingestion = DataIngestion()
    train_data_path, test_data_path = ingestion.initiate_data_ingestion()
    
    print("Data Ingestion Successful!")
    print(f"Train data saved at: {train_data_path}")
    print(f"Test data saved at: {test_data_path}")
    
except Exception as e:
    print("An error occurred:")
    print(e)

Data Ingestion Successful!
Train data saved at: artifacts\train.csv
Test data saved at: artifacts\test.csv


In [9]:
%%writefile src/Airbnb/components/data_transformation.py
import os
import sys
import pandas as pd
import numpy as np
import pickle
import re
from dataclasses import dataclass

from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from src.Airbnb.logger import logging
from src.Airbnb.exception import CustomException

@dataclass
class DataTransformationConfig:
    preprocessor_obj_file_path: str = os.path.join('artifacts', 'preprocessor.pkl')
    pricing_data_path: str = os.path.join('artifacts', 'cleaned_data_pricing.csv')
    ratings_data_path: str = os.path.join('artifacts', 'cleaned_data_ratings.csv')

class DataTransformation:
    def __init__(self):
        self.data_transformation_config = DataTransformationConfig()
        
        self.basic_essentials = [
            'Wireless Internet', 'Air conditioning', 'Kitchen', 
            'TV', 'Heating', 'Washer', 'Dryer'
        ]
        self.premium_essentials = [
            'Free parking on premises', 'Elevator', 'Indoor fireplace', 
            'Pets allowed', 'Self Check-In', '24-hour check-in', 
            'Laptop friendly workspace', 'Hot tub', 'Pool', 'Gym'
        ]

    def clean_amenities(self, x):
        if pd.isna(x):
            return []
        x = x.replace('{', '').replace('}', '').replace('"', '')
        x = re.sub(r'translation missing:[^,]+', '', x)
        return [a.strip() for a in x.split(',') if a.strip()]

    def custom_cleaning_pipeline(self, df, knn_model=None, is_train=True):
        logging.info("Applying mode imputation for bedrooms, bathrooms, and beds.")
        df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms'].mode()[0])
        df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].mode()[0])
        df['beds'] = df['beds'].fillna(df['beds'].mode()[0])

        logging.info("Extracting basic and premium amenities.")
        df['amenities_list'] = df['amenities'].apply(self.clean_amenities)
        df['basic_amenities_count'] = df['amenities_list'].apply(
            lambda x: sum(1 for item in x if item in self.basic_essentials)
        )
        df['premium_amenities_count'] = df['amenities_list'].apply(
            lambda x: sum(1 for item in x if item in self.premium_essentials)
        )
        df = df.drop(columns=['amenities', 'amenities_list'])

        logging.info("Applying Spatial KNN for missing neighbourhoods.")
        if is_train:
            known_df = df[df['neighbourhood'].notnull()]
            X_train = known_df[['latitude', 'longitude']]
            y_train = known_df['neighbourhood']
            
            knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance')
            knn_model.fit(X_train, y_train)
        
        unknown_mask = df['neighbourhood'].isnull()
        if unknown_mask.sum() > 0:
            X_test = df.loc[unknown_mask, ['latitude', 'longitude']]
            predicted_neighbourhoods = knn_model.predict(X_test)
            df.loc[unknown_mask, 'neighbourhood'] = predicted_neighbourhoods

        logging.info("Dropping defined columns and null rows.")
        columns_to_drop = ['thumbnail_url', 'first_review', 'last_review', 'zipcode']
        df = df.drop(columns=columns_to_drop)
        df = df.dropna(subset=['host_has_profile_pic', 'host_identity_verified', 'host_since'])

        logging.info("Formatting host metrics.")
        df['host_response_rate'] = df['host_response_rate'].astype(str).str.replace('%', '')
        df['host_response_rate'] = pd.to_numeric(df['host_response_rate'], errors='coerce')
        df['host_response_rate'] = df['host_response_rate'].fillna(df['host_response_rate'].median())

        binary_mapping = {'t': 1, 'f': 0}
        df['host_has_profile_pic'] = df['host_has_profile_pic'].map(binary_mapping).fillna(0)
        df['host_identity_verified'] = df['host_identity_verified'].map(binary_mapping).fillna(0)
        df['instant_bookable'] = df['instant_bookable'].map(binary_mapping).fillna(0)

        return df, knn_model

    def get_data_transformer_object(self):
        try:
            numerical_columns = [
                'accommodates', 'bathrooms', 'bedrooms', 'beds', 'latitude', 'longitude', 
                'number_of_reviews', 'review_scores_rating', 'basic_amenities_count', 
                'premium_amenities_count', 'host_response_rate', 'has_reviews',
                'host_has_profile_pic', 'host_identity_verified', 'instant_bookable'
            ]
            
            categorical_columns = ['room_type', 'property_type', 'cancellation_policy', 'city']

            num_pipeline = Pipeline(steps=[("scaler", StandardScaler())])
            
            cat_pipeline = Pipeline(steps=[
                ("one_hot_encoder", OneHotEncoder(drop='first', handle_unknown="ignore"))
            ])

            preprocessor = ColumnTransformer(
                transformers=[
                    ("num_pipeline", num_pipeline, numerical_columns),
                    ("cat_pipeline", cat_pipeline, categorical_columns)
                ],
                sparse_threshold=0  # THE FIX: Forces dense array output for np.c_
            )
            return preprocessor
            
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_data_transformation(self, train_path, test_path):
        try:
            train_df = pd.read_csv(train_path)
            test_df = pd.read_csv(test_path)
            
            logging.info("Applying custom logic to Train Set")
            train_df, trained_knn = self.custom_cleaning_pipeline(train_df, is_train=True)
            
            logging.info("Applying custom logic to Test Set")
            test_df, _ = self.custom_cleaning_pipeline(test_df, knn_model=trained_knn, is_train=False)

            df_ratings = train_df.copy()
            df_ratings = df_ratings.dropna(subset=['review_scores_rating'])
            df_ratings = df_ratings[df_ratings['number_of_reviews'] >= 3]
            os.makedirs(os.path.dirname(self.data_transformation_config.ratings_data_path), exist_ok=True)
            df_ratings.to_csv(self.data_transformation_config.ratings_data_path, index=False)

            def prepare_pricing(df):
                df_pricing = df.copy()
                df_pricing['has_reviews'] = (df_pricing['number_of_reviews'] > 0).astype(int)
                df_pricing['review_scores_rating'] = df_pricing['review_scores_rating'].fillna(df_pricing['review_scores_rating'].median())
                columns_to_drop_for_ml = ['id', 'name', 'description', 'host_since', 'neighbourhood']
                df_pricing = df_pricing.drop(columns=columns_to_drop_for_ml, errors='ignore')
                return df_pricing

            train_pricing = prepare_pricing(train_df)
            test_pricing = prepare_pricing(test_df)
            
            train_pricing.to_csv(self.data_transformation_config.pricing_data_path, index=False)

            logging.info("Obtaining preprocessing object")
            preprocessing_obj = self.get_data_transformer_object()

            target_column_name = "log_price"

            input_feature_train_df = train_pricing.drop(columns=[target_column_name])
            target_feature_train_df = train_pricing[target_column_name]

            input_feature_test_df = test_pricing.drop(columns=[target_column_name])
            target_feature_test_df = test_pricing[target_column_name]

            input_feature_train_arr = preprocessing_obj.fit_transform(input_feature_train_df)
            input_feature_test_arr = preprocessing_obj.transform(input_feature_test_df)

            train_arr = np.c_[input_feature_train_arr, np.array(target_feature_train_df)]
            test_arr = np.c_[input_feature_test_arr, np.array(target_feature_test_df)]

            os.makedirs(os.path.dirname(self.data_transformation_config.preprocessor_obj_file_path), exist_ok=True)
            with open(self.data_transformation_config.preprocessor_obj_file_path, "wb") as file_obj:
                pickle.dump(preprocessing_obj, file_obj)

            return (
                train_arr,
                test_arr,
                self.data_transformation_config.preprocessor_obj_file_path,
            )
        except Exception as e:
            raise CustomException(e, sys)

Overwriting src/Airbnb/components/data_transformation.py


In [10]:
from src.Airbnb.components.data_transformation import DataTransformation
from src.Airbnb.logger import logging

try:
    logging.info("Starting Custom Data Transformation Test...")
    
    # Paths from your data ingestion output
    train_path = 'artifacts/train.csv'
    test_path = 'artifacts/test.csv'
    
    # Initialize and run
    data_transformation = DataTransformation()
    train_arr, test_arr, preprocessor_path = data_transformation.initiate_data_transformation(train_path, test_path)
    
    print("Data Transformation Successful!")
    print(f"Train Array Shape: {train_arr.shape}")
    print(f"Test Array Shape: {test_arr.shape}")
    print(f"Preprocessor saved at: {preprocessor_path}")
    
except Exception as e:
    print("An error occurred:")
    print(e)

Data Transformation Successful!
Train Array Shape: (59128, 60)
Test Array Shape: (14795, 60)
Preprocessor saved at: artifacts\preprocessor.pkl


C:\Users\saish\End-to-End-Airbnb-Price-Prediction\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [11]:
! pip install xgboost

In [12]:
%%writefile src/Airbnb/components/model_trainer.py
import os
import sys
import pickle
from dataclasses import dataclass

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV

from src.Airbnb.logger import logging
from src.Airbnb.exception import CustomException

@dataclass
class ModelTrainerConfig:
    trained_model_file_path: str = os.path.join("artifacts", "model.pkl")

class ModelTrainer:
    def __init__(self):
        self.model_trainer_config = ModelTrainerConfig()

    def evaluate_models(self, X_train, y_train, X_test, y_test, models, params):
        try:
            report = {}
            for i in range(len(list(models))):
                model_name = list(models.keys())[i]
                model = list(models.values())[i]
                param = params[model_name]

                if param:
                    logging.info(f"Tuning hyperparameters for {model_name}...")
                    rs = RandomizedSearchCV(
                        estimator=model, 
                        param_distributions=param, 
                        n_iter=5, 
                        cv=3, 
                        n_jobs=-1, 
                        verbose=1,
                        random_state=42
                    )
                    rs.fit(X_train, y_train)
                    model.set_params(**rs.best_params_)
                
                logging.info(f"Training final {model_name}...")
                model.fit(X_train, y_train)
                
                y_test_pred = model.predict(X_test)
                test_model_score = r2_score(y_test, y_test_pred)
                report[model_name] = test_model_score
                
            return report
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_model_trainer(self, train_array, test_array):
        try:
            logging.info("Splitting training and test input data")
            
            X_train, y_train = train_array[:, :-1], train_array[:, -1]
            X_test, y_test = test_array[:, :-1], test_array[:, -1]

            models = {
                "Linear Regression": LinearRegression(),
                "Ridge Regression": Ridge(),
                "Decision Tree Regressor": DecisionTreeRegressor(),
                "Random Forest Regressor": RandomForestRegressor(random_state=42),
                "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42),
                "XGBoost Regressor": XGBRegressor(random_state=42)
            }

            params = {
                "Decision Tree Regressor": {
                    'max_depth': [10, 20, 30, None],
                    'min_samples_split': [2, 5, 10]
                },
                "Random Forest Regressor": {
                    'n_estimators': [50, 100, 200],
                    'max_depth': [10, 20, 30, None]
                },
                "Gradient Boosting Regressor": {
                    'learning_rate': [0.01, 0.05, 0.1],
                    'n_estimators': [50, 100, 200],
                    'subsample': [0.8, 0.9, 1.0]
                },
                "Linear Regression": {},
                "Ridge Regression": {
                    'alpha': [0.1, 1.0, 10.0, 100.0]
                },
                "XGBoost Regressor": {
                    'learning_rate': [0.01, 0.05, 0.1, 0.2],
                    'n_estimators': [50, 100, 200, 300],
                    'max_depth': [3, 5, 7, 9],
                    'subsample': [0.8, 0.9, 1.0]
                }
            }

            logging.info("Training and evaluating models...")
            model_report: dict = self.evaluate_models(
                X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test, 
                models=models, params=params
            )

            best_model_score = max(sorted(model_report.values()))
            best_model_name = list(model_report.keys())[
                list(model_report.values()).index(best_model_score)
            ]
            
            best_model = models[best_model_name]

            if best_model_score < 0.6:
                logging.warning("No best model found with an R-squared score above 0.60")

            logging.info(f"Best Model Found: {best_model_name} | R-squared Score: {best_model_score}")

            os.makedirs(os.path.dirname(self.model_trainer_config.trained_model_file_path), exist_ok=True)
            with open(self.model_trainer_config.trained_model_file_path, "wb") as file_obj:
                pickle.dump(best_model, file_obj)
            
            # THE FIX: Now returning the model_report dictionary as well
            return best_model_score, best_model_name, model_report
            
        except Exception as e:
            raise CustomException(e, sys)

Overwriting src/Airbnb/components/model_trainer.py


In [13]:
from src.Airbnb.components.model_trainer import ModelTrainer
from src.Airbnb.logger import logging
import warnings
warnings.filterwarnings('ignore')

try:
    logging.info("Starting Hyperparameter Tuning. This will take a few minutes as it tests dozens of combinations...")
    
    model_trainer = ModelTrainer()
    
    # Catching the third variable: model_report
    r2_score_value, best_model_name, model_report = model_trainer.initiate_model_trainer(train_arr, test_arr)
    
    print("\n" + "="*45)
    print("📊 MODEL LEADERBOARD 📊")
    print("="*45)
    
    # Sort the dictionary by R2 score in descending order
    sorted_report = dict(sorted(model_report.items(), key=lambda item: item[1], reverse=True))
    
    for model, score in sorted_report.items():
        print(f"{model:<30} | R2: {score:.4f}")

    print("\n" + "="*45)
    print("🏆 TUNING & BEST MODEL SAVED SUCCESSFUL! 🏆")
    print("="*45)
    print(f"The Winning Algorithm: {best_model_name}")
    print(f"Optimized R-squared:   {r2_score_value:.4f}")
    print(f"Model saved as:        artifacts/model.pkl")
    
except Exception as e:
    print("An error occurred:")
    print(e)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Fitting 3 folds for each of 5 candidates, totalling 15 fits
Fitting 3 folds for each of 5 candidates, totalling 15 fits
Fitting 3 folds for each of 5 candidates, totalling 15 fits
Fitting 3 folds for each of 5 candidates, totalling 15 fits

📊 MODEL LEADERBOARD 📊
XGBoost Regressor              | R2: 0.7115
Random Forest Regressor        | R2: 0.6996
Gradient Boosting Regressor    | R2: 0.6704
Decision Tree Regressor        | R2: 0.6358
Linear Regression              | R2: 0.5931
Ridge Regression               | R2: 0.5881

🏆 TUNING & BEST MODEL SAVED SUCCESSFUL! 🏆
The Winning Algorithm: XGBoost Regressor
Optimized R-squared:   0.7115
Model saved as:        artifacts/model.pkl


In [14]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

print("Loading ratings data...")
# 1. Load the Ratings Dataset
ratings_df = pd.read_csv('artifacts/cleaned_data_ratings.csv')

# Drop columns that are not useful for predicting ratings
columns_to_drop = ['id', 'name', 'description', 'host_since', 'neighbourhood', 'has_reviews']
ratings_df = ratings_df.drop(columns=columns_to_drop, errors='ignore')

# 2. Define Features (X) and Target (y)
# Our target is the review score, and we can use log_price as a feature now!
X = ratings_df.drop(columns=['review_scores_rating'])
y = ratings_df['review_scores_rating']

# 3. Build the Preprocessor
numerical_columns = X.select_dtypes(exclude="object").columns.tolist()
categorical_columns = X.select_dtypes(include="object").columns.tolist()

num_pipeline = Pipeline(steps=[("scaler", StandardScaler())])
cat_pipeline = Pipeline(steps=[("one_hot_encoder", OneHotEncoder(drop='first', handle_unknown="ignore"))])

rating_preprocessor = ColumnTransformer(
    transformers=[
        ("num_pipeline", num_pipeline, numerical_columns),
        ("cat_pipeline", cat_pipeline, categorical_columns)
    ],
    sparse_threshold=0
)

# 4. Train/Test Split & Transform
print("Preprocessing features...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_arr = rating_preprocessor.fit_transform(X_train)
X_test_arr = rating_preprocessor.transform(X_test)

# Save the rating preprocessor
os.makedirs("artifacts", exist_ok=True)
with open("artifacts/rating_preprocessor.pkl", "wb") as f:
    pickle.dump(rating_preprocessor, f)

# 5. Define Models & Tuning Parameters
models = {
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42)
}

params = {
    "Random Forest": {'n_estimators': [50, 100], 'max_depth': [10, 20, None]},
    "Gradient Boosting": {'learning_rate': [0.05, 0.1], 'n_estimators': [50, 100]},
    "XGBoost": {'learning_rate': [0.05, 0.1], 'max_depth': [3, 5, 7], 'n_estimators': [50, 100]}
}

# 6. Train and Tune
print("Training and tuning models (this may take a minute)...")
model_report = {}
for model_name, model in models.items():
    print(f"  -> Tuning {model_name}...")
    rs = RandomizedSearchCV(model, params[model_name], n_iter=3, cv=3, n_jobs=-1, random_state=42)
    rs.fit(X_train_arr, y_train)
    
    best_model = rs.best_estimator_
    y_pred = best_model.predict(X_test_arr)
    model_report[model_name] = {'model': best_model, 'score': r2_score(y_test, y_pred)}

# 7. Select Winner and Save
best_model_name = max(model_report, key=lambda k: model_report[k]['score'])
best_model = model_report[best_model_name]['model']
best_score = model_report[best_model_name]['score']

with open("artifacts/rating_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("\n" + "="*45)
print("🏆 RATINGS MODEL TRAINING SUCCESSFUL! 🏆")
print("="*45)
print(f"The Winning Algorithm: {best_model_name}")
print(f"Optimized R-squared:   {best_score:.4f}")
print("Artifacts saved:       artifacts/rating_model.pkl & artifacts/rating_preprocessor.pkl")

Loading ratings data...
Preprocessing features...
Training and tuning models (this may take a minute)...
  -> Tuning Random Forest...
  -> Tuning Gradient Boosting...
  -> Tuning XGBoost...

🏆 RATINGS MODEL TRAINING SUCCESSFUL! 🏆
The Winning Algorithm: XGBoost
Optimized R-squared:   0.1284
Artifacts saved:       artifacts/rating_model.pkl & artifacts/rating_preprocessor.pkl


In [16]:
import os
os.makedirs("src/Airbnb/pipeline", exist_ok=True)
print("Pipeline folder created successfully!")

Pipeline folder created successfully!


In [17]:
%%writefile src/Airbnb/pipeline/predict_pipeline.py
import os
import sys
import pandas as pd
import numpy as np
import pickle
from src.Airbnb.exception import CustomException
from src.Airbnb.logger import logging

class PredictPipeline:
    def __init__(self):
        pass

    def predict(self, features):
        try:
            model_path = os.path.join("artifacts", "model.pkl")
            preprocessor_path = os.path.join("artifacts", "preprocessor.pkl")
            
            # Load the saved model and preprocessor rules
            with open(model_path, 'rb') as f:
                model = pickle.load(f)
            with open(preprocessor_path, 'rb') as f:
                preprocessor = pickle.load(f)
                
            # Clean and scale the user's input data
            data_scaled = preprocessor.transform(features)
            
            # Make the prediction (this outputs a log_price)
            log_prediction = model.predict(data_scaled)
            
            # Reverse the logarithm using np.exp() to get real dollars
            actual_price_prediction = np.exp(log_prediction)
            
            return actual_price_prediction
            
        except Exception as e:
            raise CustomException(e, sys)


class CustomData:
    def __init__(self,
        accommodates: int,
        bathrooms: float,
        bedrooms: float,
        beds: float,
        latitude: float,
        longitude: float,
        number_of_reviews: int,
        review_scores_rating: float,
        basic_amenities_count: int,
        premium_amenities_count: int,
        host_response_rate: float,
        has_reviews: int,
        host_has_profile_pic: int,
        host_identity_verified: int,
        instant_bookable: int,
        room_type: str,
        property_type: str,
        cancellation_policy: str,
        city: str
    ):
        self.accommodates = accommodates
        self.bathrooms = bathrooms
        self.bedrooms = bedrooms
        self.beds = beds
        self.latitude = latitude
        self.longitude = longitude
        self.number_of_reviews = number_of_reviews
        self.review_scores_rating = review_scores_rating
        self.basic_amenities_count = basic_amenities_count
        self.premium_amenities_count = premium_amenities_count
        self.host_response_rate = host_response_rate
        self.has_reviews = has_reviews
        self.host_has_profile_pic = host_has_profile_pic
        self.host_identity_verified = host_identity_verified
        self.instant_bookable = instant_bookable
        self.room_type = room_type
        self.property_type = property_type
        self.cancellation_policy = cancellation_policy
        self.city = city

    def get_data_as_data_frame(self):
        try:
            # Package the user's inputs into a dictionary
            custom_data_input_dict = {
                "accommodates": [self.accommodates],
                "bathrooms": [self.bathrooms],
                "bedrooms": [self.bedrooms],
                "beds": [self.beds],
                "latitude": [self.latitude],
                "longitude": [self.longitude],
                "number_of_reviews": [self.number_of_reviews],
                "review_scores_rating": [self.review_scores_rating],
                "basic_amenities_count": [self.basic_amenities_count],
                "premium_amenities_count": [self.premium_amenities_count],
                "host_response_rate": [self.host_response_rate],
                "has_reviews": [self.has_reviews],
                "host_has_profile_pic": [self.host_has_profile_pic],
                "host_identity_verified": [self.host_identity_verified],
                "instant_bookable": [self.instant_bookable],
                "room_type": [self.room_type],
                "property_type": [self.property_type],
                "cancellation_policy": [self.cancellation_policy],
                "city": [self.city],
            }

            # Convert dictionary to a Pandas DataFrame
            return pd.DataFrame(custom_data_input_dict)

        except Exception as e:
            raise CustomException(e, sys)

Writing src/Airbnb/pipeline/predict_pipeline.py


In [2]:
%%writefile app.py
import xgboost
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import os
from src.Airbnb.pipeline.predict_pipeline import CustomData, PredictPipeline

# Set up visual configuration
st.set_page_config(page_title="Airbnb Price Predictor", page_icon="🏠", layout="wide")

st.title("🏠 Airbnb Nightly Price Predictor")
st.markdown("Enter property details below to calculate an estimated nightly price using our optimized XGBoost machine learning model.")
st.markdown("---")

# Interface Columns
col1, col2, col3 = st.columns(3)

with col1:
    st.header("📍 Location & Type")
    city = st.selectbox("City", ["NYC", "LA", "SF", "DC", "Chicago", "Boston"])
    room_type = st.selectbox("Room Type", ["Entire home/apt", "Private room", "Shared room"])
    property_type = st.selectbox("Property Type", ["Apartment", "House", "Condominium", "Loft", "Townhouse", "Other"])
    latitude = st.number_input("Latitude", value=40.7128, format="%.4f")
    longitude = st.number_input("Longitude", value=-74.0060, format="%.4f")

with col2:
    st.header("🛏️ Property Details")
    accommodates = st.number_input("Accommodates (Guests)", min_value=1, max_value=16, value=2)
    bedrooms = st.number_input("Bedrooms", min_value=0.0, max_value=10.0, value=1.0, step=0.5)
    beds = st.number_input("Beds", min_value=0.0, max_value=20.0, value=1.0, step=1.0)
    bathrooms = st.number_input("Bathrooms", min_value=0.0, max_value=10.0, value=1.0, step=0.5)
    basic_amenities_count = st.number_input("Basic Amenities Count", min_value=0, max_value=50, value=10)
    premium_amenities_count = st.number_input("Premium Amenities Count", min_value=0, max_value=20, value=2)

with col3:
    st.header("⭐ Host & Listing Info")
    cancellation_policy = st.selectbox("Cancellation Policy", ["flexible", "moderate", "strict", "super_strict_30", "super_strict_60"])
    review_scores_rating = st.slider("Review Score Rating (0-100)", min_value=0, max_value=100, value=95)
    number_of_reviews = st.number_input("Number of Reviews", min_value=0, max_value=1000, value=10)
    host_response_rate = st.slider("Host Response Rate (%)", min_value=0.0, max_value=1.0, value=1.0, step=0.01)
    
    st.markdown("**Check all that apply:**")
    has_reviews_input = st.checkbox("Listing Has Reviews?", value=True)
    host_has_profile_pic_input = st.checkbox("Host Has Profile Pic?", value=True)
    host_identity_verified_input = st.checkbox("Host Identity Verified?", value=True)
    instant_bookable_input = st.checkbox("Instant Bookable?", value=False)

# Convert boolean inputs to binary numerical values
has_reviews = 1 if has_reviews_input else 0
host_has_profile_pic = 1 if host_has_profile_pic_input else 0
host_identity_verified = 1 if host_identity_verified_input else 0
instant_bookable = 1 if instant_bookable_input else 0

st.markdown("---")

# Predict Button
if st.button("💰 Predict Nightly Price", type="primary", use_container_width=True):
    with st.spinner("Running XGBoost Algorithm..."):
        try:
            data = CustomData(
                accommodates=accommodates, bathrooms=bathrooms, bedrooms=bedrooms,
                beds=beds, latitude=latitude, longitude=longitude,
                number_of_reviews=number_of_reviews, review_scores_rating=review_scores_rating,
                basic_amenities_count=basic_amenities_count, premium_amenities_count=premium_amenities_count,
                host_response_rate=host_response_rate, has_reviews=has_reviews,
                host_has_profile_pic=host_has_profile_pic, host_identity_verified=host_identity_verified,
                instant_bookable=instant_bookable, room_type=room_type,
                property_type=property_type, cancellation_policy=cancellation_policy, city=city
            )
            
            pred_df = data.get_data_as_data_frame()
            predict_pipeline = PredictPipeline()
            results = predict_pipeline.predict(pred_df)
            
            st.success("Prediction Complete!")
            st.metric(label="Calculated Optimal Nightly Price", value=f"${results[0]:.2f}")
            
        except Exception as e:
            st.error(f"An error occurred: {str(e)}")

Overwriting app.py
